In [1]:
import pandas as pd
import numpy as np

historical_df = pd.read_csv(
    "../data/interim/historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"]
)

historical_df = historical_df.sort_values(
    "date"
).reset_index(drop=True)

In [2]:
teams = pd.concat([
    historical_df["home_team"],
    historical_df["away_team"]
]).unique()

elo_ratings = {
    team: 1500
    for team in teams
}

In [3]:
def expected_score(rating_a, rating_b):
    return 1 / (
        1 + 10 ** (
            (rating_b - rating_a) / 400
        )
    )

In [4]:
def actual_score(goals_for, goals_against):

    if goals_for > goals_against:
        return 1

    elif goals_for < goals_against:
        return 0

    return 0.5

In [5]:
home_elo_before = []
away_elo_before = []

In [6]:
K = 20

for _, row in historical_df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    home_rating = elo_ratings[home]
    away_rating = elo_ratings[away]

    home_elo_before.append(home_rating)
    away_elo_before.append(away_rating)

    expected_home = expected_score(
        home_rating,
        away_rating
    )

    expected_away = expected_score(
        away_rating,
        home_rating
    )

    actual_home = actual_score(
        row["home_score"],
        row["away_score"]
    )

    actual_away = 1 - actual_home

    elo_ratings[home] = (
        home_rating
        + K * (actual_home - expected_home)
    )

    elo_ratings[away] = (
        away_rating
        + K * (actual_away - expected_away)
    )

In [7]:
historical_df["home_elo"] = home_elo_before
historical_df["away_elo"] = away_elo_before

historical_df["elo_diff"] = (
    historical_df["home_elo"]
    - historical_df["away_elo"]
)

In [8]:
historical_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_elo",
        "away_elo",
        "elo_diff"
    ]
].tail()

,date,home_team,away_team,home_elo,away_elo,elo_diff
49399,2026-06-09,Ethiopia,Malawi,1421.544524,1423.639825,-2.095301
49400,2026-06-10,England,Costa Rica,1895.868768,1651.238336,244.630431
49401,2026-06-10,Portugal,Nigeria,1896.115777,1721.529086,174.586692
49402,2026-06-10,Bolivia,Algeria,1581.078620,1784.491812,-203.413192
49403,2026-06-10,Afghanistan,Pakistan,1285.117095,1179.386710,105.730385


In [9]:
sorted(
    elo_ratings.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]


[('Argentina', 1988.7592098702264),
 ('Spain', 1981.4263594018025),
 ('France', 1939.421028372195),
 ('Brazil', 1918.657582399574),
 ('Portugal', 1901.4749603468028),
 ('England', 1899.7991110925332),
 ('Germany', 1888.3382965251487),
 ('Colombia', 1883.9872921245153),
 ('Netherlands', 1872.298405949542),
 ('Japan', 1861.6759637267558),
 ('Italy', 1852.55819794156),
 ('Morocco', 1851.0522014757385),
 ('Belgium', 1849.626469453052),
 ('Croatia', 1836.9434125754522),
 ('Ecuador', 1826.8946640893691),
 ('Mexico', 1823.067294155654),
 ('Uruguay', 1817.2211653090121),
 ('Denmark', 1805.1854023532014),
 ('Iran', 1804.6395645883422),
 ('Switzerland', 1796.885909125456)]

In [10]:
import pickle

with open(
    "../models/elo_ratings.pkl",
    "wb"
) as f:

    pickle.dump(
        elo_ratings,
        f
    )

print("Archivo guardado correctamente")

Archivo guardado correctamente


In [11]:
historical_df.to_csv(
    "../data/processed/historical_matches_with_elo.csv",
    index=False
)

print("Archivo guardado correctamente")

Archivo guardado correctamente
